# 03 — Preprocess & Train

## Goal
Load enriched features from `02_feature_engineering.ipynb`, apply model-specific
preprocessing, run optional Optuna hyperparameter tuning, and save all artifacts
needed by `04_predict_evaluate.ipynb`.

**This notebook is a pure orchestrator — all logic lives in `src/pipeline_preprocess.py`.**

## Inputs
```
outputs/enriched/train_enriched.parquet  — enriched features (no target)
outputs/enriched/y_train.parquet         — target aligned by index
```

## Outputs
```
outputs/preproc/X_train_lgbm.parquet    — label-encoded train features (LightGBM / XGBoost)
outputs/preproc/X_val_lgbm.parquet      — label-encoded val features   (LightGBM / XGBoost)
outputs/preproc/X_train_raw.parquet     — raw train features BEFORE encoding (CatBoost)
outputs/preproc/X_val_raw.parquet       — raw val features BEFORE encoding   (CatBoost)
outputs/preproc/y_train.parquet         — train target (split portion)
outputs/preproc/y_val.parquet           — val target
outputs/preproc/encoding_map.pkl        — fitted label encoders (for test)
outputs/best_params_lgbm.json           — best Optuna params for LightGBM
outputs/best_params_xgb.json            — best Optuna params for XGBoost
```

## Step 0 — Setup

In [1]:
import sys
import os
import warnings
warnings.filterwarnings('ignore')

# Notebook lives in v2/ — go one level up to reach project root
PROJECT_ROOT_PATH = os.path.abspath(os.path.join(os.getcwd(), '..'))
for subdir in ['', 'src', 'v0']:
    p = os.path.join(PROJECT_ROOT_PATH, subdir)
    if p not in sys.path:
        sys.path.insert(0, p)

from config import (
    PROJECT_ROOT, NON_FEATURE_COLS,
    ENRICHED_DIR, PREPROC_DIR, OUTPUTS_DIR
)
from pipeline_preprocess import (
    load_enriched,
    split_train_val,
    preprocess_and_save,
    load_preprocessed,
    run_optuna_lgbm,
)
from tune_optuna import tune_xgb

print(f'Project root : {PROJECT_ROOT}')
print(f'Enriched dir : {ENRICHED_DIR}')
print(f'Preproc dir  : {PREPROC_DIR}')


Project root : C:\Users\USER\Desktop\NOTEBOOKS\HIT\ML_Fraud_detection
Enriched dir : C:\Users\USER\Desktop\NOTEBOOKS\HIT\ML_Fraud_detection\outputs\enriched
Preproc dir  : C:\Users\USER\Desktop\NOTEBOOKS\HIT\ML_Fraud_detection\outputs\preproc


## Step 1 — Run control flags

### Preprocessing flags
- `PREPROC_READY_LGBM_XGB = True` — skip LightGBM / XGBoost preprocessing, load saved files

> **Note:** LightGBM and XGBoost share identical preprocessing (label encoding + fill NaN).
> They use the same files: `X_train_lgbm.parquet` / `X_val_lgbm.parquet`.


In [2]:
# ── Preprocessing flags ───────────────────────────────────────────────────
# LightGBM and XGBoost share identical preprocessing (label encoding + fill NaN).
# They use the same saved files: X_train_lgbm.parquet / X_val_lgbm.parquet.
PREPROC_READY_LGBM_XGB = True   # Set True to skip preprocessing and load saved files directly.

# ── Optuna flags ───────────────────────────────────────────────────────────
RUN_OPTUNA_LGBM = False   # True -> tune LightGBM, save best_params_lgbm.json
RUN_OPTUNA_XGB  = False    # True -> tune XGBoost,  save best_params_xgb.json

# ── Quality / latency profile ──────────────────────────────────────────────
# "min"  — ~0.7h       | ~0.004 AUC loss vs high
# "med"  — ~3h         | ~0.001 AUC loss vs high
# "med_high" — ~8-10h  | ~0.0005 AUC loss vs high
# "high" — ~18h        | reference (no loss)
QUALITY = "med_high"

print(f'PREPROC_READY_LGBM_XGB : {PREPROC_READY_LGBM_XGB}')
print(f'RUN_OPTUNA_LGBM        : {RUN_OPTUNA_LGBM}')
print(f'RUN_OPTUNA_XGB         : {RUN_OPTUNA_XGB}')
print(f'QUALITY                : {QUALITY}')


PREPROC_READY_LGBM_XGB : True
RUN_OPTUNA_LGBM        : False
RUN_OPTUNA_XGB         : False
QUALITY                : med_high


## Step 2 — Load enriched data

In [3]:
if not PREPROC_READY_LGBM_XGB:
    train_enriched, y_train_full = load_enriched(ENRICHED_DIR)
else:
    print('PREPROC_READY_LGBM_XGB=True — skipping data load.')


PREPROC_READY_LGBM_XGB=True — skipping data load.


## Step 3 — Time-based train/val split (80/20 by TransactionDT)

In [4]:
if not PREPROC_READY_LGBM_XGB:
    X_train, X_val, y_train, y_val = split_train_val(train_enriched, y_train_full)
else:
    print('PREPROC_READY_LGBM_XGB=True — skipping split.')


PREPROC_READY_LGBM_XGB=True — skipping split.


## Step 4 — Model-specific preprocessing

Each model has its own preprocessing pipeline and its own `PREPROC_READY` flag.
Set the flag to `True` to skip preprocessing and load saved files directly.

In [5]:
import pandas as pd

# ── Step 4: LightGBM / XGBoost preprocessing ──────────────────────────────
# LightGBM and XGBoost share identical preprocessing: label encoding + fill NaN.
# Output: X_train_lgbm.parquet, X_val_lgbm.parquet, encoding_map.pkl
if not PREPROC_READY_LGBM_XGB:
    X_train_lgbm, X_val_lgbm, encoding_map = preprocess_and_save(
        X_train, X_val, y_train, y_val,
        preproc_dir =PREPROC_DIR,
        cols_to_drop=NON_FEATURE_COLS,
    )
else:
    X_train_lgbm = pd.read_parquet(PREPROC_DIR / 'X_train_lgbm.parquet')
    X_val_lgbm   = pd.read_parquet(PREPROC_DIR / 'X_val_lgbm.parquet')
    y_train      = pd.read_parquet(PREPROC_DIR / 'y_train.parquet')['isFraud']
    y_val        = pd.read_parquet(PREPROC_DIR / 'y_val.parquet')['isFraud']
    import pickle
    with open(PREPROC_DIR / 'encoding_map.pkl', 'rb') as f:
        encoding_map = pickle.load(f)
    print(f'Loaded X_train_lgbm {X_train_lgbm.shape}, X_val_lgbm {X_val_lgbm.shape}')


Loaded X_train_lgbm (472432, 497), X_val_lgbm (118108, 497)


## Step 5 — Optuna hyperparameter tuning

Runs Bayesian TPE search for each model independently.
Each model runs only if its `RUN_OPTUNA_*` flag is `True`.
Best params are saved to JSON and loaded automatically by `train_*.py`.

**Quality profiles** (set `QUALITY` in Step 1):
| Profile | Trials | Data fraction | Time    | AUC loss vs high |
|---------|--------|---------------|---------|------------------|
| `min`   | 30     | 30%           | ~0.7h   | ~0.004           |
| `med`   | 50     | 50%           | ~3h     | ~0.001           |
| `high`  | 100    | 100%          | ~18h    | 0 (reference)    |

### Step 5.1 — Optuna: LightGBM

In [6]:
if RUN_OPTUNA_LGBM:
    run_optuna_lgbm(
        X_train_lgbm, y_train,
        X_val_lgbm,   y_val,
        outputs_dir=OUTPUTS_DIR,
        quality=QUALITY,
    )
else:
    params_path = OUTPUTS_DIR / 'best_params_lgbm.json'
    if params_path.exists():
        print(f'RUN_OPTUNA_LGBM=False — using saved params:\n  {params_path}')
    else:
        print('RUN_OPTUNA_LGBM=False and no saved params found.')
        print('train_lightgbm.py will use DEFAULT_PARAMS.')
        print(f'Expected: {params_path}')

RUN_OPTUNA_LGBM=False — using saved params:
  C:\Users\USER\Desktop\NOTEBOOKS\HIT\ML_Fraud_detection\outputs\best_params_lgbm.json


### Step 5.2 — Optuna: XGBoost

In [7]:
if RUN_OPTUNA_XGB:
    tune_xgb(
        X_train_lgbm, y_train,
        X_val_lgbm,   y_val,
        quality=QUALITY,
        save_path=str(OUTPUTS_DIR / 'best_params_xgb.json'),
    )
else:
    params_path = OUTPUTS_DIR / 'best_params_xgb.json'
    if params_path.exists():
        print(f'RUN_OPTUNA_XGB=False — using saved params:\n  {params_path}')
    else:
        print('RUN_OPTUNA_XGB=False and no saved params found.')
        print('train_xgboost.py will use DEFAULT_PARAMS.')
        print(f'Expected: {params_path}')

RUN_OPTUNA_XGB=False — using saved params:
  C:\Users\USER\Desktop\NOTEBOOKS\HIT\ML_Fraud_detection\outputs\best_params_xgb.json


## Step 6 — Summary

In [8]:
print('=' * 60)
print('PREPROCESSING SUMMARY')
print('=' * 60)
print(f'  X_train_lgbm : {X_train_lgbm.shape}  | fraud rate: {y_train.mean():.4%}')
print(f'  X_val_lgbm   : {X_val_lgbm.shape}    | fraud rate: {y_val.mean():.4%}')
print(f'  encoding_map : {len(encoding_map)} label encoders fitted on train')
print()
print('  Dtypes in X_train_lgbm:')
for dtype, count in X_train_lgbm.dtypes.value_counts().items():
    print(f'    {str(dtype):<12}: {count} columns')
print()

# NaN check — after preprocessing there must be zero NaN
nan_train = X_train_lgbm.isna().sum().sum()
nan_val   = X_val_lgbm.isna().sum().sum()
print(f'  NaN in X_train_lgbm : {nan_train} (expected: 0)')
print(f'  NaN in X_val_lgbm   : {nan_val}   (expected: 0)')

assert nan_train == 0, 'NaN found in X_train_lgbm after preprocessing!'
assert nan_val   == 0, 'NaN found in X_val_lgbm after preprocessing!'
print('  NaN check: OK ✓')
print()

# Confirm saved files exist
saved_files = [
    PREPROC_DIR / 'X_train_lgbm.parquet',
    PREPROC_DIR / 'X_val_lgbm.parquet',
    PREPROC_DIR / 'y_train.parquet',
    PREPROC_DIR / 'y_val.parquet',
    PREPROC_DIR / 'encoding_map.pkl',
    OUTPUTS_DIR / 'best_params_lgbm.json',
    OUTPUTS_DIR / 'best_params_xgb.json',
]
print('  Saved files:')
for path in saved_files:
    status = '✓' if path.exists() else '✗ MISSING'
    print(f'    {status}  {path}')
print('=' * 60)


PREPROCESSING SUMMARY
  X_train_lgbm : (472432, 497)  | fraud rate: 3.5135%
  X_val_lgbm   : (118108, 497)    | fraud rate: 3.4409%
  encoding_map : 47 label encoders fitted on train

  Dtypes in X_train_lgbm:
    float32     : 444 columns
    int32       : 49 columns
    int64       : 2 columns
    int16       : 1 columns
    int8        : 1 columns

  NaN in X_train_lgbm : 0 (expected: 0)
  NaN in X_val_lgbm   : 0   (expected: 0)
  NaN check: OK ✓

  Saved files:
    ✓  C:\Users\USER\Desktop\NOTEBOOKS\HIT\ML_Fraud_detection\outputs\preproc\X_train_lgbm.parquet
    ✓  C:\Users\USER\Desktop\NOTEBOOKS\HIT\ML_Fraud_detection\outputs\preproc\X_val_lgbm.parquet
    ✓  C:\Users\USER\Desktop\NOTEBOOKS\HIT\ML_Fraud_detection\outputs\preproc\y_train.parquet
    ✓  C:\Users\USER\Desktop\NOTEBOOKS\HIT\ML_Fraud_detection\outputs\preproc\y_val.parquet
    ✓  C:\Users\USER\Desktop\NOTEBOOKS\HIT\ML_Fraud_detection\outputs\preproc\encoding_map.pkl
    ✓  C:\Users\USER\Desktop\NOTEBOOKS\HIT\ML_Fraud_d